## Female

In [6]:
import pandas as pd

df_female = pd.read_csv(
    "data/raw/enrollment/SECONDARY_Female_School_Enrollment_DATA.csv",
    skiprows=4
)

df_female.to_sql(
    "secondary_female_raw",
    conn,
    if_exists="replace",
    index=False
)

266

In [7]:
pd.read_sql("SELECT COUNT(*) FROM secondary_female_raw;", conn)

,COUNT(*)
0,266


In [8]:
import sqlite3
conn = sqlite3.connect("data/education.db")

cursor = conn.execute("""
SELECT name FROM sqlite_master WHERE type='table';
""")

cursor.fetchall()

[('secondary_male_raw',), ('secondary_female_raw',)]

In [9]:
conn.executescript("""
DROP TABLE IF EXISTS secondary_female_clean;
DROP TABLE IF EXISTS secondary_male_clean;
""")

In [10]:
conn.executescript("""
CREATE TABLE secondary_female_clean AS
SELECT
    "Country Name" AS country_name,
    "Country Code" AS country_code,
    "Indicator Name" AS indicator_name,

    "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009",
    "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019",
    "2020","2021","2022","2023"

FROM secondary_female_raw

WHERE "Country Code" IN (
'AFE','AFW','ARB','AUS','EAS','EUU','LCN','NAC','SAS',
'LIC','LMC','UMC','HIC'
);
""")

In [11]:
pd.read_sql("SELECT COUNT(*) FROM secondary_female_clean;", conn)

,COUNT(*)
0,13


## Male

In [17]:
import pandas as pd

df_male = pd.read_csv(
    "data/raw/enrollment/SECONDARY_Male_School_Enrollment_DATA.csv",
    skiprows=4
)

In [18]:
df_male.head()

,Country Name,Country Code,Indicator Name,Indicator Code,1960,1961,1962,1963,1964,1965,...,2017,2018,2019,2020,2021,2022,2023,2024,2025,Unnamed: 70
0,Aruba,ABW,"School enrollment, secondary, male (% gross)",SE.SEC.ENRR.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,131.166290,133.686264,127.464012,113.229683,126.545250,NaN,119.946092,NaN,NaN
1,Africa Eastern and Southern,AFE,"School enrollment, secondary, male (% gross)",SE.SEC.ENRR.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,45.650219,45.961262,46.588360,47.018871,47.512051,47.630058,NaN,NaN,NaN,NaN
2,Afghanistan,AFG,"School enrollment, secondary, male (% gross)",SE.SEC.ENRR.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,70.354591,72.566017,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Africa Western and Central,AFW,"School enrollment, secondary, male (% gross)",SE.SEC.ENRR.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,45.040970,44.882309,45.678570,46.097710,46.703461,46.314171,46.760971,46.526329,NaN,NaN
4,Angola,AGO,"School enrollment, secondary, male (% gross)",SE.SEC.ENRR.MA,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,56.833503,NaN,54.839228,NaN,NaN,NaN


In [19]:
df_male_clean = df_male[
    ["Country Name", "Country Code", "Indicator Name",
     "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009",
     "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019",
     "2020","2021","2022","2023"]
]

In [20]:
df_male_clean = df_male_clean.dropna(how="all", subset=[
    "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009",
    "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019",
    "2020","2021","2022","2023"
])

In [21]:
df_male_clean.to_sql(
    "secondary_male_raw",
    conn,
    if_exists="replace",
    index=False
)

248

In [27]:
conn.executescript("""
DROP TABLE IF EXISTS secondary_male_clean;

CREATE TABLE secondary_male_clean AS
SELECT
    "Country Name" AS country_name,
    TRIM("Country Code") AS country_code,
    "Indicator Name" AS indicator_name,

    "2000","2001","2002","2003","2004","2005","2006","2007","2008","2009",
    "2010","2011","2012","2013","2014","2015","2016","2017","2018","2019",
    "2020","2021","2022","2023"

FROM secondary_male_raw
WHERE TRIM("Country Code") IN (
'AFE','AFW','ARB','AUS','EAS','EUU','LCN','NAC','SAS',
'LIC','LMC','UMC','HIC'
);
""")

In [28]:
pd.read_sql("SELECT COUNT(*) FROM secondary_male_clean;", conn)

,COUNT(*)
0,13


In [68]:
pd.read_sql("""
SELECT
  'male' AS table_name,
  COUNT(*) AS rows,
  MIN("2000") AS min_2000,
  MAX("2000") AS max_2000
FROM secondary_male_clean

UNION ALL

SELECT
  'female',
  COUNT(*),
  MIN("2000"),
  MAX("2000")
FROM secondary_female_clean;
""", conn)

,table_name,rows,min_2000,max_2000
0,male,13,25.832979,101.213089
1,female,13,18.782869,101.815292


In [69]:
import os
os.listdir("data/cleaned")

['.gitkeep', 'secondary_female_clean.csv', 'secondary_male_clean.csv']

In [70]:
conn.executescript("""
DROP TABLE IF EXISTS male_long_sql;
DROP TABLE IF EXISTS female_long_sql;

CREATE TABLE male_long_sql AS
SELECT country_name, country_code, '2000' AS year, "2000" AS male_enrollment FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2001', "2001" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2002', "2002" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2003', "2003" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2004', "2004" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2005', "2005" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2006', "2006" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2007', "2007" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2008', "2008" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2009', "2009" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2010', "2010" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2011', "2011" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2012', "2012" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2013', "2013" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2014', "2014" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2015', "2015" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2016', "2016" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2017', "2017" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2018', "2018" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2019', "2019" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2020', "2020" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2021', "2021" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2022', "2022" FROM secondary_male_clean
UNION ALL SELECT country_name, country_code, '2023', "2023" FROM secondary_male_clean;

CREATE TABLE female_long_sql AS
SELECT country_name, country_code, '2000' AS year, "2000" AS female_enrollment FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2001', "2001" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2002', "2002" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2003', "2003" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2004', "2004" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2005', "2005" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2006', "2006" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2007', "2007" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2008', "2008" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2009', "2009" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2010', "2010" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2011', "2011" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2012', "2012" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2013', "2013" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2014', "2014" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2015', "2015" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2016', "2016" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2017', "2017" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2018', "2018" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2019', "2019" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2020', "2020" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2021', "2021" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2022', "2022" FROM secondary_female_clean
UNION ALL SELECT country_name, country_code, '2023', "2023" FROM secondary_female_clean;
""")

In [71]:
conn.executescript("""
DROP TABLE IF EXISTS secondary_gender_merged;

CREATE TABLE secondary_gender_merged AS
SELECT
    m.country_name,
    m.country_code,
    m.year,
    m.male_enrollment,
    f.female_enrollment
FROM male_long_sql m
JOIN female_long_sql f
    ON m.country_code = f.country_code
    AND m.year = f.year;
""")

In [72]:
pd.read_sql("SELECT COUNT(*) FROM secondary_gender_merged;", conn)

,COUNT(*)
0,312


In [73]:
pd.read_sql("SELECT * FROM secondary_gender_merged LIMIT 10;", conn)

,country_name,country_code,year,male_enrollment,female_enrollment
0,Africa Eastern and Southern,AFE,2000,29.914789,25.548540
1,Africa Western and Central,AFW,2000,25.832979,18.782869
2,Arab World,ARB,2000,62.048500,56.196072
3,Australia,AUS,2000,NaN,NaN
4,East Asia & Pacific,EAS,2000,63.266891,60.945709
5,European Union,EUU,2000,101.213089,101.815292
6,High income,HIC,2000,97.895401,99.302322
7,Latin America & Caribbean,LCN,2000,77.025833,81.789207
8,Low income,LIC,2000,28.755510,19.253220
9,Lower middle income,LMC,2000,49.151890,39.224239


In [ ]:
df = pd.read_sql("SELECT * FROM secondary_gender_merged;", conn)

df.to_csv("data/cleaned/secondary_gender_merged.csv", index=False)